# META-CXR Spark NLG Evaluation on Kaggle

Computes BLEU-1/2/3/4, METEOR, ROUGE-L in parallel using PySpark (local mode)  
across all 7 encoder runs. Input: JSONL pred/ref pairs from GCS.  
Output: summary CSV + JSON → uploaded to GCS.

**GCS paths**
- Input:  `gs://meta-cxr-checkpoint/eval/BERTSCore/Vicuna/reports_vicuna_{run}.jsonl`
- Output: `gs://meta-cxr-checkpoint/eval/spark_nlg_metrics/`

## Cell 0 — Load Kaggle Secrets

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["GCS_SERVICE_ACCOUNT"] = user_secrets.get_secret("GCS_SERVICE_ACCOUNT")
print("Kaggle secret loaded: GCS_SERVICE_ACCOUNT")

## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys

packages = [
    "pyspark==3.5.1",
    "google-cloud-storage",
    "nltk",
    "pycocoevalcap",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("punkt", quiet=True)
print("Dependencies ready.")

## Cell 2 — GCS Auth + Download JSONL Files

In [ ]:
import base64
import json
import os
from pathlib import Path
from google.cloud import storage
from google.oauth2 import service_account

GCS_PROJECT       = "mimic-cxr-jpg-491409"
GCS_BUCKET        = "meta-cxr-checkpoint"
GCS_INPUT_PREFIX  = "eval/BERTSCore/Vicuna"
GCS_OUTPUT_PREFIX = "eval/spark_nlg_metrics"

LOCAL_JSONL_DIR  = Path("/kaggle/temp/jsonl")
LOCAL_OUTPUT_DIR = Path("/kaggle/working/spark_nlg_output")
LOCAL_JSONL_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUNS = [
    "01_biovil_only",
    "02_pubmedclip_only",
    "03_swin_only",
    "04_biovil_pubmedclip",
    "05_biovil_swin",
    "06_pubmedclip_swin",
    "07_all_three",
]

ENCODER_INFO = {
    "01_biovil_only":       {"RN50": "+", "ViT": "-", "Swin": "-"},
    "02_pubmedclip_only":   {"RN50": "-", "ViT": "+", "Swin": "-"},
    "03_swin_only":         {"RN50": "-", "ViT": "-", "Swin": "+"},
    "04_biovil_pubmedclip": {"RN50": "+", "ViT": "+", "Swin": "-"},
    "05_biovil_swin":       {"RN50": "+", "ViT": "-", "Swin": "+"},
    "06_pubmedclip_swin":   {"RN50": "-", "ViT": "+", "Swin": "+"},
    "07_all_three":         {"RN50": "+", "ViT": "+", "Swin": "+"},
}


def _build_gcs_client():
    raw = os.environ.get("GCS_SERVICE_ACCOUNT", "").strip()
    try:
        info = json.loads(raw)
    except json.JSONDecodeError:
        info = json.loads(base64.b64decode(raw).decode())
    creds = service_account.Credentials.from_service_account_info(info)
    return storage.Client(project=GCS_PROJECT, credentials=creds)


gcs_client = _build_gcs_client()
bucket = gcs_client.bucket(GCS_BUCKET)

downloaded = []
for run in RUNS:
    blob_name  = f"{GCS_INPUT_PREFIX}/reports_vicuna_{run}.jsonl"
    local_path = LOCAL_JSONL_DIR / f"reports_vicuna_{run}.jsonl"
    blob = bucket.blob(blob_name)
    if blob.exists():
        blob.download_to_filename(str(local_path))
        downloaded.append(run)
        print(f"  downloaded: {blob_name}")
    else:
        print(f"  SKIP (not found): {blob_name}")

print(f"\nDownloaded {len(downloaded)}/{len(RUNS)} JSONL files.")

## Cell 3 — Start Spark (local mode)

In [ ]:
import os
from pyspark.sql import SparkSession

N_CORES = os.cpu_count() or 2

spark = (
    SparkSession.builder
    .master(f"local[{N_CORES}]")
    .appName("MetaCXR-SparkNLGEval")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", str(N_CORES * 4))
    .getOrCreate()
)

print(f"Spark {spark.version} running — local[{N_CORES}]")

## Cell 4 — Load JSONL → Spark DataFrame

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

rows = []
for run in downloaded:
    path = LOCAL_JSONL_DIR / f"reports_vicuna_{run}.jsonl"
    with open(path) as f:
        for sample_id, line in enumerate(f):
            obj = json.loads(line)
            rows.append((run, sample_id, str(obj["pred"]), str(obj["ref"])))

schema = StructType([
    StructField("run",       StringType(),  False),
    StructField("sample_id", IntegerType(), False),
    StructField("pred",      StringType(),  True),
    StructField("ref",       StringType(),  True),
])

df = spark.createDataFrame(rows, schema=schema).repartition(N_CORES * 4)

print(f"Total rows: {df.count()}")
df.show(3, truncate=60)

## Cell 5 — Định nghĩa pandas UDFs (per-sample, chạy song song)

In [ ]:
import re
import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import FloatType


def _tok(text: str):
    return re.findall(r"\w+", str(text).lower())


@pandas_udf(FloatType())
def bleu1_udf(preds: pd.Series, refs: pd.Series) -> pd.Series:
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    smooth = SmoothingFunction().method1
    out = []
    for p, r in zip(preds, refs):
        pt, rt = _tok(p), [_tok(r)]
        out.append(float(sentence_bleu(rt, pt, weights=(1,0,0,0), smoothing_function=smooth)) if pt else 0.0)
    return pd.Series(out)


@pandas_udf(FloatType())
def bleu2_udf(preds: pd.Series, refs: pd.Series) -> pd.Series:
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    smooth = SmoothingFunction().method1
    out = []
    for p, r in zip(preds, refs):
        pt, rt = _tok(p), [_tok(r)]
        out.append(float(sentence_bleu(rt, pt, weights=(.5,.5,0,0), smoothing_function=smooth)) if pt else 0.0)
    return pd.Series(out)


@pandas_udf(FloatType())
def bleu3_udf(preds: pd.Series, refs: pd.Series) -> pd.Series:
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    smooth = SmoothingFunction().method1
    out = []
    for p, r in zip(preds, refs):
        pt, rt = _tok(p), [_tok(r)]
        out.append(float(sentence_bleu(rt, pt, weights=(1/3,1/3,1/3,0), smoothing_function=smooth)) if pt else 0.0)
    return pd.Series(out)


@pandas_udf(FloatType())
def bleu4_udf(preds: pd.Series, refs: pd.Series) -> pd.Series:
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    smooth = SmoothingFunction().method1
    out = []
    for p, r in zip(preds, refs):
        pt, rt = _tok(p), [_tok(r)]
        out.append(float(sentence_bleu(rt, pt, weights=(.25,.25,.25,.25), smoothing_function=smooth)) if pt else 0.0)
    return pd.Series(out)


@pandas_udf(FloatType())
def meteor_udf(preds: pd.Series, refs: pd.Series) -> pd.Series:
    from nltk.translate.meteor_score import meteor_score
    out = []
    for p, r in zip(preds, refs):
        pt, rt = _tok(p), _tok(r)
        try:
            out.append(float(meteor_score([rt], pt)) if pt and rt else 0.0)
        except Exception:
            out.append(0.0)
    return pd.Series(out)


@pandas_udf(FloatType())
def rougel_udf(preds: pd.Series, refs: pd.Series) -> pd.Series:
    def _lcs(a, b):
        m, n = len(a), len(b)
        prev = [0] * (n + 1)
        for i in range(1, m + 1):
            curr = [0] * (n + 1)
            for j in range(1, n + 1):
                curr[j] = prev[j-1]+1 if a[i-1]==b[j-1] else max(prev[j], curr[j-1])
            prev = curr
        return prev[n]

    out = []
    for p, r in zip(preds, refs):
        pt, rt = _tok(p), _tok(r)
        if not pt or not rt:
            out.append(0.0); continue
        lcs  = _lcs(pt, rt)
        prec = lcs / len(pt)
        rec  = lcs / len(rt)
        f1   = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
        out.append(float(f1))
    return pd.Series(out)


print("UDFs ready: bleu1, bleu2, bleu3, bleu4, meteor, rougel")

## Cell 6 — Tính song song + Aggregate theo run

In [ ]:
from pyspark.sql.functions import col, avg, count

df_scored = (
    df
    .withColumn("bleu1",  bleu1_udf(col("pred"), col("ref")))
    .withColumn("bleu2",  bleu2_udf(col("pred"), col("ref")))
    .withColumn("bleu3",  bleu3_udf(col("pred"), col("ref")))
    .withColumn("bleu4",  bleu4_udf(col("pred"), col("ref")))
    .withColumn("meteor", meteor_udf(col("pred"), col("ref")))
    .withColumn("rougel", rougel_udf(col("pred"), col("ref")))
)

summary_spark = (
    df_scored.groupBy("run")
    .agg(
        avg("bleu1").alias("BLEU-1"),
        avg("bleu2").alias("BLEU-2"),
        avg("bleu3").alias("BLEU-3"),
        avg("bleu4").alias("BLEU-4"),
        avg("meteor").alias("METEOR"),
        avg("rougel").alias("ROUGE-L"),
        count("*").alias("n_samples"),
    )
    .orderBy("run")
)

summary_spark.show(truncate=False)

## Cell 7 — CIDEr (corpus-level, per run) + Merge

In [ ]:
from pycocoevalcap.cider.cider import Cider

cider_scores = {}
for run in downloaded:
    path = LOCAL_JSONL_DIR / f"reports_vicuna_{run}.jsonl"
    preds_map, refs_map = {}, {}
    with open(path) as f:
        for i, line in enumerate(f):
            obj = json.loads(line)
            preds_map[i] = [str(obj["pred"])]
            refs_map[i]  = [str(obj["ref"])]
    score, _ = Cider().compute_score(refs_map, preds_map)
    cider_scores[run] = round(float(score), 4)
    print(f"  {run}: CIDEr = {cider_scores[run]}")

summary_pd = summary_spark.toPandas()

for col_name in ["BLEU-1","BLEU-2","BLEU-3","BLEU-4","METEOR","ROUGE-L"]:
    summary_pd[col_name] = summary_pd[col_name].round(4)

summary_pd["CIDEr"] = summary_pd["run"].map(cider_scores)
summary_pd["RN50"]  = summary_pd["run"].map(lambda r: ENCODER_INFO[r]["RN50"])
summary_pd["ViT"]   = summary_pd["run"].map(lambda r: ENCODER_INFO[r]["ViT"])
summary_pd["Swin"]  = summary_pd["run"].map(lambda r: ENCODER_INFO[r]["Swin"])

col_order = ["run","RN50","ViT","Swin",
             "BLEU-1","BLEU-2","BLEU-3","BLEU-4",
             "METEOR","ROUGE-L","CIDEr","n_samples"]
summary_pd = summary_pd[col_order]

print("\nFinal summary:")
print(summary_pd.to_string(index=False))

## Cell 8 — Save + Upload to GCS

In [ ]:
csv_path  = LOCAL_OUTPUT_DIR / "spark_nlg_summary.csv"
json_path = LOCAL_OUTPUT_DIR / "spark_nlg_summary.json"

summary_pd.to_csv(csv_path, index=False)

json_out = {
    "method": "spark_local",
    "n_cores": N_CORES,
    "runs": summary_pd.to_dict(orient="records"),
}
with open(json_path, "w") as f:
    json.dump(json_out, f, indent=2)

print(f"Saved locally: {csv_path}, {json_path}")

# Upload
for local_path in [csv_path, json_path]:
    blob_name = f"{GCS_OUTPUT_PREFIX}/{local_path.name}"
    bucket.blob(blob_name).upload_from_filename(str(local_path))
    print(f"  gs://{GCS_BUCKET}/{blob_name}")

spark.stop()
print("Done.")